# Fake-account detection on Cresci-2017 and Cresci-2015: experiments on Kaggle CPU

**What this is.** A fresh set of experiments for a new paper: baselines (LightGBM, XGBoost, Random Forest, GCN,
GraphSAGE, GAT, R-GCN), a metadata + text + GCN + LightGBM hybrid baseline, the proposed RGF model, its ablations,
robustness, calibration and cross-dataset tests. Every number comes from this run. Every result row carries a
provenance label: `THIS-WORK`, `REPRODUCED-CORRECTED` (the hybrid baseline), `WALKTHROUGH` (single seed, never a
result) or `SMOKE-TEST-NOT-A-RESULT`. TwiBot-20 is not used.

**Settings (right panel):** Accelerator **None** (CPU). Internet is **not needed** if the text model is attached as
an input (item 5 below); with Internet **On** it is downloaded instead.

**Inputs (Add Input):**
1. `botrecon_repo.zip`: this code (private dataset).
2. Cresci-2017: download from the Botometer bot repository (same page as Cresci-2015) and upload it as a **private** Kaggle dataset. Do NOT use the Kaggle copy `hashemalsalmi/cresci-2017`: it holds only 2 of the 9 groups.
3. Cresci-2015: download from the Botometer bot repository (https://botometer.osome.iu.edu/bot-repository/datasets.html)
   and upload it as a **private** Kaggle dataset (zip or tar.gz is fine; it is extracted automatically). It must contain
   the group folders E13, TFP, FSF, INT, TWT with `users.csv`, `followers.csv`, `friends.csv`. **Optional**: without
   it everything runs on Cresci-2017 and the Cresci-2015 parts run when you add it later.
4. From the second session on: the `botrecon_results.zip` saved at the end of the previous session.
5. Text model, for notebooks without internet: the Kaggle dataset `inversion/sentence-transformers-222`
   (contains `all-MiniLM-L6-v2`, a git clone of the official model; its commit is recorded in every result row).

**How to run:** *Run All*. Every experiment cell stops starting new work before the session budget is used up and
**resumes** next session. For one unattended run, set `SESSION_BUDGET_MIN = 600` in step 1 and use
*Save Version → Save & Run All*. Dataset paths are detected automatically; to override them, edit
`config/kaggle_cpu.json` (no path edits in the notebook).

**After a kernel restart:** run step 1 again, then any later cell; each cell re-reads what it needs from disk.

## 1. Environment setup

In [ ]:
SESSION_BUDGET_MIN = None    # None = use config/kaggle_cpu.json (40 min); 600 for an unattended Save & Run All
import glob, os, shutil, sys
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "botrecon_repo")
src = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/botrecon/__init__.py", recursive=True)]
if src:
    assert len(src) == 1, f"More than one copy of the code is attached as input: {src}. Remove the old one."
    for sub in ("botrecon", "tests", "notebooks", "config"):          # always refresh the code from the input;
        shutil.rmtree(os.path.join(REPO, sub), ignore_errors=True)   # results/ and cache/ are kept
    shutil.copytree(src[0], REPO, dirs_exist_ok=True)
assert os.path.exists(os.path.join(REPO, "botrecon")), "Add botrecon_repo.zip as an input (Add Input -> your private dataset)"
print("code version:", open(os.path.join(REPO, "VERSION")).read().strip() if os.path.exists(os.path.join(REPO, "VERSION")) else "unknown")
os.chdir(REPO); sys.path.insert(0, REPO)
# restore results/caches/checkpoints from a previous session's botrecon_results.zip, if added as an input
prev = sorted(glob.glob("/kaggle/input/**/botrecon_repo/results/suite_main.jsonl", recursive=True) +
              glob.glob("/kaggle/input/**/botrecon_repo/results/runs.csv", recursive=True))
if prev and not os.path.exists("results/.restored"):
    base = os.path.dirname(os.path.dirname(prev[0]))
    for sub in ("results", "cache"):
        if os.path.exists(os.path.join(base, sub)):
            shutil.copytree(os.path.join(base, sub), sub, dirs_exist_ok=True)
    os.makedirs("results", exist_ok=True); open("results/.restored", "w").write(base)
    print("restored previous session from", base)
from botrecon.notebook import start_session
from botrecon.settings import load_config
start_session(SESSION_BUDGET_MIN or load_config("config/kaggle_cpu.json")["session_budget_minutes"])
print("repository:", REPO)

## 2. Dependency installation and tests
All tests must pass, including those that train the real models on synthetic data (about 2–3 min). If anything fails, stop and send this cell's output.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
import importlib.util, subprocess
if importlib.util.find_spec("sentence_transformers") is None:
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], capture_output=True, text=True)
    if pip.returncode != 0:
        raise SystemExit("Could not install sentence-transformers. Turn Internet ON in the notebook settings "
                         "(needs a phone-verified Kaggle account), then run this cell again.\n" + pip.stderr[-800:])
from botrecon.env import check_environment
env = check_environment(strict=True)
print({k: env[k] for k in ("python", "cpu_count", "ram_gb", "disk_free_gb", "gpu")}); print(env["packages"])
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rs", "-p", "no:cacheprovider"], capture_output=True, text=True)
open("results/pytest.txt", "w").write(r.stdout + r.stderr)
print("\n".join((r.stdout + r.stderr).strip().splitlines()[-15:]))
if r.returncode != 0:
    raise SystemExit("Tests failed: stop here and send this output.")

## 3. Configuration

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
import json
from botrecon.notebook import ctx
c = ctx()
print(json.dumps(c.cfg, indent=1)); print("session budget:", c.budget, "min; left:", c.left(), "min; threads:", c.threads)

## 4. Dataset path verification

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon.notebook import ctx
c = ctx(refresh_paths=True)
print(json.dumps(c.paths, indent=1))
OFFICIAL = "https://botometer.osome.iu.edu/bot-repository/datasets.html"
if not c.paths.get("cresci2017"):
    raise SystemExit(f"Cresci-2017 not found. Download cresci-2017 from {OFFICIAL}, upload it as a private "
                     "Kaggle dataset and add it as an input.")
found = c.paths.get("groups_found", {})
if found.get("cresci2017", 9) < 9:
    raise SystemExit(f"Cresci-2017 copy is INCOMPLETE: {found['cresci2017']} of 9 groups. The Kaggle copy "
                     f"hashemalsalmi/cresci-2017 lacks most groups. Download cresci-2017 from {OFFICIAL}, upload it "
                     "as a private dataset, add it as an input and remove the incomplete one.")
if c.paths.get("cresci2015") and found.get("cresci2015", 5) < 5:
    raise SystemExit(f"Cresci-2015 copy is INCOMPLETE: {found['cresci2015']} of 5 groups.")
print("datasets available:", c.available, "" if c.paths.get("cresci2015") else "(Cresci-2015 not added yet: Cresci-2017 only)")
# text model: a local copy (Kaggle input) is used if attached; otherwise it is downloaded (needs Internet ON)
import socket
from botrecon.text import local_model_path
model = CF.corrected("cresci2017").text_model_b
local = local_model_path(model, marker="modules.json")
def _online():
    try:
        socket.create_connection(("huggingface.co", 443), timeout=3).close(); return True
    except OSError:
        return False
if local:
    print("text model: local copy", local)
elif _online():
    print("text model: will be downloaded from huggingface.co")
else:
    raise SystemExit("No internet and no local copy of the text model. Add Input -> search "
                     "'inversion/sentence-transformers-222' -> add it, then run again (or turn Internet ON).")

## 5. Dataset inspection
Group counts are compared with the published sizes; this settles the manuscript's 11,952-account question for the data you actually have.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import config as CF
from botrecon.notebook import ctx
from botrecon.run import cached_load
from botrecon.text import empty_rates, text_inputs
c = ctx(); audit = {}
for ds in c.available:
    d = cached_load(CF.corrected(ds), c.paths[ds], "cache", False)
    audit[ds] = {**d.log, "bot_share": round(float(d.y.mean()), 4), "empty_bio": empty_rates(text_inputs(d, "B"), d.y)}
    del d
json.dump(audit, open("results/data_audit.json", "w"), indent=1, default=str)
print(json.dumps(audit, indent=1, default=str)[:6000])

## 6. Label validation

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
import pandas as pd
from botrecon.audit import label_audit
from botrecon.notebook import ctx
c = ctx(); rows = []
for ds in c.available:
    rows += [{"dataset": ds, **r} for r in label_audit(cached_load(CF.corrected(ds), c.paths[ds], "cache", False))]
json.dump(rows, open("results/label_audit.json", "w"), indent=1)
pd.set_option("display.max_colwidth", 140); display(pd.DataFrame(rows))

## 7. Train/validation/test splitting and leakage checks
Cresci-2017 and Cresci-2015: stratified 70/10/20 per seed (neither has an official split). Any FAIL other than a published-size mismatch stops the notebook.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon.audit import protocol_audit
from botrecon.notebook import ctx
c = ctx(); rows = []
for ds in c.available:
    d = cached_load(CF.corrected(ds), c.paths[ds], "cache", False)
    rows += [{"dataset": ds, **r} for r in protocol_audit(d, seed=0, text_variant=c.cfg["text_variant"])]
json.dump(rows, open("results/protocol_audit.json", "w"), indent=1)
display(pd.DataFrame(rows))
blocking = [r for r in rows if r["status"] == "FAIL" and "published sizes" not in r["check"]]
if any(r["status"] == "FAIL" and "published sizes" in r["check"] for r in rows):
    print("WARNING: group sizes differ from the published release; report the actual loaded counts in the paper.")
if blocking:
    raise SystemExit(f"Protocol audit failed: {blocking}")

## 8. Metadata processing
Four features named in the manuscript (account age, follower ratio, tweet frequency, username entropy); formulas are reconstruction choices. Median imputation, signed log1p and z-scoring are fitted on training rows only.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_metadata(c), indent=1, default=float)[:5000])

## 9. Text embedding generation
Variant B (default): `sentence-transformers/all-MiniLM-L6-v2` on profile descriptions, mean pooling, 384-d; empty text gives a zero vector. Variant A (BERT-base [CLS] on up to 20 tweets, 128 tokens) runs only if `text_variant` is `A` or `run_bert_tweets` is true. Embeddings are cached in `cache/` and reused by every experiment.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_text(c), indent=1, default=float)[:5000])

## 10. Graph construction
Cresci-2015: follower and friend edges among labelled accounts, undirected. Cresci-2017 (no graph released): kNN (k = 10, cosine) on standardised metadata, symmetrised. Transductive: test accounts' features and edges are present, their labels never.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_graph(c), indent=1, default=float)[:5000])

## 11. GCN training
GCN(d → 64) → ReLU → Dropout(0.5) → GCN(64 → 2), AdamW (lr 0.01, weight decay 5e-4), ≤ 100 epochs, model with the lowest validation loss. Trained on training half A (blending).

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_gcn(c), indent=1, default=float)[:5000])

## 12. Embedding extraction
Two different outputs, never interchanged: the 64-d hidden embedding and the 2-d class probabilities.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_embeddings(c), indent=1, default=float)[:5000])

## 13. LightGBM training: P1 and P3
**P3** = GCN 64-d embedding → LightGBM. **P1** = metadata + text + GCN 64-d embedding → LightGBM. Both are trained on training half B and early-stopped on validation.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_lightgbm(c), indent=1, default=float)[:5000])

## 14. Evaluation
Bot-class, macro and weighted metrics, MCC, ROC-AUC, PR-AUC, confusion counts; internal consistency is checked. Single seed: labelled WALKTHROUGH, not a reported result.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon import walkthrough as W
from botrecon.notebook import ctx
c = ctx(); print("dataset:", c.walk_dataset, "| seed:", c.walk_seed)
print(json.dumps(W.step_evaluate(c), indent=1, default=float)[:5000])

## 15. Baseline experiments
Transductive suite, 5 seeds: LightGBM (metadata / text / both), XGBoost, Random Forest, GCN, GraphSAGE, GAT, BotRGCN-style R-GCN (Cresci-2015), the reconstructed hybrid in four variants (P1/P3 × 64-d/2-d), the proposed RGF, and the modality ablations Metadata + GCN and Text + GCN. Robustness, calibration, cross-dataset (Cresci-2017 ↔ Cresci-2015, shared accounts removed by user ID) and explainability measurements are recorded for seeds 0–2. Resumable.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
import shlex, subprocess
from botrecon.notebook import ctx
def plan(part, reserve):
    c = ctx()
    minutes = c.left(reserve)
    if minutes < 1:
        print(f"no session time left for '{part}'; run the notebook again next session"); return
    args = ["--part", part, "--cresci", c.paths["cresci2017"], "--minutes", str(minutes), "--threads", str(c.threads),
            "--seeds", *map(str, c.cfg["seeds"]), "--robust-seeds", *map(str, c.cfg["robust_seeds"]),
            "--checkpoint-seeds", *map(str, c.cfg["checkpoint_seeds"]), "--text", c.cfg["text_variant"]]
    if c.paths.get("cresci2015"):
        args += ["--cresci2015", c.paths["cresci2015"]]
    if c.cfg["run_bert_tweets"]:
        args += ["--with-bert-tweets"]
    print("python -m botrecon.plan " + shlex.join(args))
    subprocess.run([sys.executable, "-m", "botrecon.plan", *args], check=True)
plan("main", reserve=6)

## 16. Ablation experiments
RGF ablations (without gate, modality dropout, auxiliary losses, each expert; concat fusion), the inductive (unseen-account) setting, and the reconstruction study (manuscript-faithful screen and the test-set early-stopping diagnostic).

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
plan("extended", reserve=4)

## 17. Statistical analysis
Mean ± SD over seeds, 95% t-intervals, paired t-tests against RGF with Holm correction (no Wilcoxon with 5 seeds). Tables are built only from finished runs.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon.suite_report import build
md_text = build("results", "results/paper")
for title in ("T2 Main comparison", "T3b Modality ablation", "T10 Paired t-tests"):
    i = md_text.find(title)
    print(md_text[i - 4: i + 2500] if i >= 0 else f"{title}: no results yet"); print()

## 18. Results export
Download `botrecon_results.zip` from the Output panel. Add it as an input next session to resume, and send it for drafting when the run is complete.

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
import zipfile
out = os.path.join(os.path.dirname(os.getcwd()), "botrecon_results.zip")   # next to botrecon_repo/
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for top in ("results", "cache"):
        for root, _, files in os.walk(top):
            for f in files:
                z.write(os.path.join(root, f), os.path.join("botrecon_repo", root, f))
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")
for f in sorted(glob.glob("results/*.json*") + glob.glob("results/*.csv") + glob.glob("results/paper/*")):
    print(" ", f)

## 19. Final reproducibility report

In [ ]:
import glob, json, os, sys
import pandas as pd
from botrecon import config as CF
from botrecon.run import cached_load
from botrecon.notebook import ctx
from botrecon.repro_report import write
print(write(ctx()))